# Valoricore × LangChain

**Deterministic vector memory with cryptographic audit trails — natively in LangChain.**

This notebook shows:
1. One-line setup — `ValoricoreLangChain` replaces FAISS / Chroma / Pinecone
2. Adding texts and documents
3. Similarity search + distance scores
4. Tag-filtered search (tenant isolation)
5. Using as a LangChain retriever in a RAG chain
6. Cryptographic state proof — BLAKE3 hash
7. Snapshot & bit-exact restore

No server required. Everything runs locally in this Colab cell.

[![GitHub](https://img.shields.io/badge/GitHub-Valori--Kernel-6c47ff?logo=github)](https://github.com/varshith-Git/Valori-Kernel)

## 1 · Install

In [ ]:
%%capture
!pip install "valoricore[langchain,local]"
# langchain  → langchain-core + langchain
# local      → sentence-transformers (offline embeddings, no API key)

## 2 · Imports & embedder

In [ ]:
from valoricore.integrations import ValoricoreLangChain
from valoricore.embeddings import SentenceTransformerEmbedder

# Offline embedder — no API key needed
# Swap with OpenAIEmbeddings() or any LangChain Embeddings object
class _Embedder:
    """Thin shim so SentenceTransformerEmbedder satisfies LangChain's interface."""
    def __init__(self, model="all-MiniLM-L6-v2"):
        self._e = SentenceTransformerEmbedder(model)
    def embed_documents(self, texts):
        return [self._e.embed(t) for t in texts]
    def embed_query(self, text):
        return self._e.embed(text)

embedding = _Embedder()
print(f"Embedding dim: {len(embedding.embed_query('test'))}")

## 3 · One-line initialization

This is all users need. No adapter objects, no HTTP server, no separate config.

In [ ]:
store = ValoricoreLangChain(
    path       = "/tmp/valori_langchain_demo",
    embedding  = embedding,
    index_kind = "bruteforce",   # "bruteforce" | "hnsw" | "ivf"
)

# To use a remote node instead:
#   store = ValoricoreLangChain(remote="http://my-node:3000", embedding=embedding)

print("Store initialized ✓")
print(f"Initial state hash: {store.get_state_hash()[:16]}...")

## 4 · Add texts

In [ ]:
texts = [
    "Valoricore is a deterministic vector database built in Rust.",
    "Q16.16 fixed-point arithmetic produces identical results on x86, ARM, and RISC-V.",
    "Every insert is backed by a BLAKE3 Merkle proof.",
    "The event log is append-only and fsynced before any live state change.",
    "LangChain users can swap FAISS for Valoricore with one line of code.",
    "LlamaIndex users can use ValoricoreLlamaIndex as a drop-in vector store.",
    "Crash recovery is mathematically proven, not just claimed.",
    "Leader-follower replication uses tokio::sync::watch for race-free coordination.",
]

metadatas = [
    {"source": "overview",     "topic": "intro"},
    {"source": "architecture", "topic": "determinism"},
    {"source": "architecture", "topic": "crypto"},
    {"source": "architecture", "topic": "durability"},
    {"source": "integrations", "topic": "langchain"},
    {"source": "integrations", "topic": "llamaindex"},
    {"source": "overview",     "topic": "crash-recovery"},
    {"source": "architecture", "topic": "replication"},
]

ids = store.add_texts(texts, metadatas)

print(f"Inserted {len(ids)} records")
print(f"Record IDs: {ids}")
print(f"State hash after insert: {store.get_state_hash()[:16]}...")

## 5 · Add LangChain Documents

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content = "The HNSW index enables sub-millisecond search at millions of records.",
        metadata     = {"source": "architecture", "topic": "hnsw"},
    ),
    Document(
        page_content = "The IVF index uses deterministic k-means with FNV-1a seeding.",
        metadata     = {"source": "architecture", "topic": "ivf"},
    ),
]

ids = store.add_documents(documents)
print(f"Inserted {len(ids)} documents, IDs: {ids}")

## 6 · Similarity search

In [ ]:
results = store.similarity_search("How does crash recovery work?", k=3)

print(f"Top {len(results)} results:\n")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")
    print(f"   metadata: {doc.metadata}")
    print()

## 7 · Similarity search with distance scores

Score is the raw Q16.16² L2 distance — **lower means more similar**.

In [ ]:
pairs = store.similarity_search_with_score("deterministic arithmetic", k=4)

print("Results with L2 distance (lower = closer):\n")
for doc, score in pairs:
    print(f"  score={score:.1f}  →  {doc.page_content[:70]}...")

## 8 · Search with a pre-computed vector

In [ ]:
# Useful when you already have an embedding from an external source
my_vector = embedding.embed_query("BLAKE3 Merkle proof")
docs = store.similarity_search_by_vector(my_vector, k=2)

for doc in docs:
    print("→", doc.page_content)

## 9 · Tag-filtered search (tenant / namespace isolation)

Tag filtering is O(1) overhead and 100% accurate — no false positives.

In [ ]:
# Insert records with explicit tag values
store._client._db.insert(embedding.embed_query("tenant A private doc"), tag=1)
store._client._db.insert(embedding.embed_query("tenant B private doc"), tag=2)
store._client.set_metadata(
    store._client._db.record_count() - 2,
    b'{"_valori_text": "tenant A private doc"}'
)

# Search restricted to tag=1 only — tenant B record never appears
docs = store.similarity_search("private document", k=5, filter_tag=1)
print(f"Results for tag=1: {len(docs)} document(s)")
for doc in docs:
    print("  →", doc.page_content or "(no text metadata)")

## 10 · Use as a LangChain retriever

Plugs directly into `RetrievalQA`, `ConversationalRetrievalChain`, agents, and any
LangChain component that accepts a `BaseRetriever`.

In [ ]:
retriever = store.as_retriever(k=3)

# Direct retrieval
docs = retriever.get_relevant_documents("fixed-point arithmetic")
print(f"Retrieved {len(docs)} docs via retriever:")
for doc in docs:
    print(f"  · {doc.page_content[:80]}")

In [ ]:
# ── RAG chain with OpenAI (requires OPENAI_API_KEY) ──────────────────────────
# Uncomment to run:

# import os
# os.environ["OPENAI_API_KEY"] = "sk-..."
#
# from langchain.chains import RetrievalQA
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
#
# openai_embedding = OpenAIEmbeddings()
# openai_store = ValoricoreLangChain(
#     path      = "/tmp/valori_openai",
#     embedding = openai_embedding,
# )
# openai_store.add_texts(texts, metadatas)
#
# chain = RetrievalQA.from_chain_type(
#     llm       = ChatOpenAI(model="gpt-4o-mini"),
#     retriever = openai_store.as_retriever(k=4),
# )
# answer = chain.run("How does Valoricore prove crash recovery?")
# print(answer)

## 11 · From texts / from documents (factory pattern)

In [ ]:
# Standard LangChain factory idiom — one call creates the store and inserts
fresh_store = ValoricoreLangChain.from_texts(
    texts     = ["factory pattern doc 1", "factory pattern doc 2"],
    embedding = embedding,
    path      = "/tmp/valori_factory",
)

docs = fresh_store.similarity_search("factory", k=2)
print(f"from_texts: {len(docs)} result(s)")

## 12 · Cryptographic state proof

The BLAKE3 root is **deterministic across all machines**. Compute it before and after
any operation to prove exactly what changed — or that nothing did.

In [ ]:
h1 = store.get_state_hash()
print(f"Hash before insert : {h1}")

store.add_texts(["new document added after hash capture"])

h2 = store.get_state_hash()
print(f"Hash after insert  : {h2}")
print(f"Hashes differ      : {h1 != h2}")

## 13 · Snapshot & bit-exact restore

In [ ]:
hash_before = store.get_state_hash()
snap        = store.snapshot()          # serialize full kernel state

print(f"Snapshot size : {len(snap):,} bytes")
print(f"Hash before   : {hash_before}")

# Restore into a fresh store
restored = ValoricoreLangChain(
    path      = "/tmp/valori_restored",
    embedding = embedding,
)
restored.restore(snap)

hash_after = restored.get_state_hash()
print(f"Hash after    : {hash_after}")
print(f"Bit-exact restore: {hash_before == hash_after} ✓")

## Summary

| What you did | Old way | With `valoricore.integrations` |
|---|---|---|
| Initialize | `ValoricoreAdapter(base_url=...) + LangChainVectorStore(adapter=...)` | `ValoricoreLangChain(path="./db", embedding=...)` |
| Add texts | same | `store.add_texts(texts, metadatas)` |
| Search | same | `store.similarity_search(query, k=5)` |
| As retriever | manual class | `store.as_retriever(k=5)` |
| Local embedded | not supported | `path="./db"` |
| State proof | not available | `store.get_state_hash()` |
| Snapshot | not available | `store.snapshot()` / `store.restore(snap)` |

**Next steps:**
- [LlamaIndex Integration notebook](https://colab.research.google.com/drive/1Q72ANZxBm1fthNpgVW-FftS8sZz6uCr3)
- [End-to-End Demo notebook](https://colab.research.google.com/drive/1QO1yQMQoGbp9fwrb00KVKTq5bYVGXgJv)
- [GitHub: Valori-Kernel](https://github.com/varshith-Git/Valori-Kernel)